# Scratch

## iseg sanity test

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if not (NOTEBOOK_DIR / "scratch.ipynb").exists():
    NOTEBOOK_DIR = (Path.cwd() / "notebooks" / "full_model").resolve()
LFM_ROOT = NOTEBOOK_DIR.parents[1]

if str(LFM_ROOT) not in sys.path:
    sys.path.insert(0, str(LFM_ROOT))

from lfm.full_model.datamodules import LunarInstanceSegmentationDatamodule
from lfm.full_model.utils import create_timestamped_output_dir, plot_instance_batch_sanity
from lfm.full_model.utils.utils import ensure_data_symlink

print("Notebook directory:", NOTEBOOK_DIR)
print("LFM root:", LFM_ROOT)


KeyboardInterrupt



KeyboardInterrupt: 

In [5]:
ISEG_DATA_ROOT = Path("")  # expects train/val/test/{chips,labels}
ISEG_OUTPUT_DIR = create_timestamped_output_dir(NOTEBOOK_DIR / "outputs" / "instance_sanity")

iseg_datamodule = LunarInstanceSegmentationDatamodule(
    data_root=ISEG_DATA_ROOT,
    crop_size=256,
    batch_size=5,
    num_workers=0,
)

plot_instance_batch_sanity(
    iseg_datamodule,
    output_dir=ISEG_OUTPUT_DIR,
    split="train",
    n_samples=5,
)

NameError: name 'create_timestamped_output_dir' is not defined

## Examine data

In [18]:
from pathlib import Path

LFM_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/")
DATA_ROOT = LFM_ROOT / "model_inputs/300_300_inputs/kaguya_static_all_wac"
ISEG = DATA_ROOT / "inst_seg"
ISEG_LABELS = ISEG / "labels"

label_files = list(ISEG_LABELS.glob("*.npz"))
example_label = label_files[0]
example_label_archive = np.load(example_label, allow_pickle=True)
label = example_label_archive
print(f"example label loaded: {example_label_archive}")
example_data = example_label_archive['mask']
example_bboxes = example_label_archive['bboxes']

print(example_data.shape, "\n", example_bboxes)

example label loaded: NpzFile '/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg/labels/M1108346760CE_r7950_c900_label.npz' with keys: mask, bboxes, num_craters
(300, 300) 
 [[283. 250.  12.  11.]
 [240. 266.  11.  11.]
 [ 40. 278.  19.  19.]
 [ 13. 261.  10.  10.]
 [192.  35.  12.  12.]
 [180.  26.  15.  16.]
 [110.  36.  14.  14.]
 [ 88. 147.  15.  15.]
 [ 62. 173.  28.  28.]
 [ 63. 206.  19.  18.]]


In [19]:
example_n_craters = example_label_archive['num_craters']
print(f"num_craters: {example_n_craters}, len(bboxes): {len(example_bboxes)}")

print(f"Unique mask values: {np.unique(label["mask"])}")

print('Datatypes of mask, bboxes, num_craters: ')
print(label["mask"].dtype)
print(label["bboxes"].dtype)
print(label["num_craters"].dtype)

num_craters: 10, len(bboxes): 10
Unique mask values: [ 0  1  2  3  4  5  6  7  8  9 10]
Datatypes of mask, bboxes, num_craters: 
uint8
float64
int64


## Create 7 band dataset
Creates `data_7band/{split}/{chips,labels}` from `data/{split}/{chips,labels}`. Chip TIFFs keep only bands 1-7. Labels are copied unchanged.

In [1]:
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
import logging
import os
import shutil

try:
    import pyproj
    os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
except Exception:
    pass

logging.getLogger("rasterio._env").setLevel(logging.ERROR)

import rasterio

SRC_ROOT = Path("data")
DST_ROOT = Path("data_7band")
MAX_WORKERS = 16


def write_7band_chip(args):
    chip_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")

        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

    return str(out_path)


def copy_label(args):
    label_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, out_path)
    return str(out_path)


def process_split(split):
    src_chips = SRC_ROOT / split / "chips"
    src_labels = SRC_ROOT / split / "labels"
    dst_chips = DST_ROOT / split / "chips"
    dst_labels = DST_ROOT / split / "labels"

    chip_jobs = [(p, dst_chips / p.name) for p in sorted(src_chips.glob("*.tif"))]
    label_jobs = [(p, dst_labels / p.name) for p in sorted(src_labels.iterdir()) if p.is_file()]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chip_outputs = list(executor.map(write_7band_chip, chip_jobs))

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        label_outputs = list(executor.map(copy_label, label_jobs))

    print(f"{split}: wrote {len(chip_outputs)} chips and copied {len(label_outputs)} labels")

In [2]:
# for split in ["train"]:
#     process_split(split)

In [3]:
# for split in ["val"]:
#     process_split(split)

In [4]:
# for split in ["test"]:
#     process_split(split)

## Create Semantic Segmentation Splits

Builds a fresh `train`/`val`/`test` split from a semantic-segmentation source directory. This does not use the existing `data` split layout.

In [5]:
import random

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "7_band_vis_uv/sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_7band_chip((chip_path, chip_out))
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [5]:
import random

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "7_band_vis_uv/sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_7band_chip((chip_path, chip_out))
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [6]:
create_sem_seg_split()

matched pairs: 623
chips only:    0
labels only:   51
train: 497 pairs
val: 26 pairs
test: 100 pairs
wrote split dataset to: /panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_sem_seg_v2


## Create Instance Segmentation Splits

Builds a fresh `train`/`val`/`test` split from the instance-segmentation source directory. Chip TIFFs are saved with only the first 7 bands. `.npz` labels are copied unchanged.

In [ ]:
import random
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import rasterio

ISEG_SOURCE_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg")
ISEG_OUTPUT_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")

ISEG_CHIPS_DIR = ISEG_SOURCE_ROOT / "chips"
ISEG_LABELS_DIR = ISEG_SOURCE_ROOT / "labels"

ISEG_IMAGE_GLOB = "*.tif"
ISEG_LABEL_GLOB = "*_label.npz"
ISEG_IMAGE_SUFFIX = "_input_wac_static_chip"
ISEG_LABEL_SUFFIX = "_label"

ISEG_SEED = 42
ISEG_N_TEST = 100
ISEG_TRAIN_FRACTION = 0.95
ISEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_iseg_pairs():
    chips = {
        split_key(path, ISEG_IMAGE_SUFFIX): path
        for path in sorted(ISEG_CHIPS_DIR.glob(ISEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, ISEG_LABEL_SUFFIX): path
        for path in sorted(ISEG_LABELS_DIR.glob(ISEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= ISEG_N_TEST:
        raise ValueError(f"Need more than {ISEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def write_iseg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_iseg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = ISEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = ISEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_iseg_7band_chip(chip_path, chip_out)
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_iseg_split():
    pairs = find_iseg_pairs()
    rng = random.Random(ISEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:ISEG_N_TEST]
    remaining = pairs[ISEG_N_TEST:]
    n_train = int(round(len(remaining) * ISEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=ISEG_MAX_WORKERS) as executor:
        list(executor.map(copy_iseg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {ISEG_OUTPUT_ROOT.resolve()}")

In [ ]:
create_iseg_split()

In [ ]:
for split in ["train", "val", "test"]:
    chips = sorted((ISEG_OUTPUT_ROOT / split / "chips").glob("*.tif"))
    labels = sorted((ISEG_OUTPUT_ROOT / split / "labels").glob("*.npz"))
    other_labels = [p for p in (ISEG_OUTPUT_ROOT / split / "labels").iterdir() if p.is_file() and p.suffix != ".npz"]
    print(f"{split}: {len(chips)} chips, {len(labels)} .npz labels, {len(other_labels)} non-npz labels")